In [1]:
# import libraries and define functions.

import pandas
import pathlib
import pydash
import requests
import unidecode

def value_extract(row, col):

    # extract dictionary values. 

    return pydash.get(row[col], 'value')    
    
def sparql_query(query, service):

    # send sparql request, and formulate results into a dataframe.

    r = requests.get(service, params = {'format': 'json', 'query': query})
    data = pydash.get(r.json(), 'results.bindings')
    data = pandas.DataFrame.from_dict(data)
    for x in data.columns:    
        data[x] = data.apply(value_extract, col=x, axis=1)
    return data

def string_norm(row, col):

  # normalise strings for matching.

  return unidecode.unidecode(row[col]).upper()

def annual_query(filt):

    # capture year period of wikidata film/creator data
    # or all films without publication date.

    annual = sparql_query("""
        SELECT DISTINCT ?creator ?creatorLabel ?work ?workLabel
        WHERE {
            ?work wdt:P31 wd:Q11424.
            """+filt+"""
            ?work ?property ?creator.
            ?creator wdt:P31 wd:Q5.
            SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }""", 'https://query.wikidata.org/sparql')
    
    if len(annual):
        annual = annual.rename(columns={
            'creator':'creator_id', 
            'creatorLabel':'creator_name',
            'work':'work_id', 
            'workLabel':'work_name',
        })

        annual = annual[['creator_id', 'creator_name', 'work_id', 'work_name']]

        for x in ['creator_id', 'work_id']:
            annual[x] = annual[x].str.split('/').str[-1]

        for x in ['creator_name', 'work_name']:
            annual[x] = annual.apply(string_norm, col=x, axis=1)

        return annual

In [2]:
# build dataframe of wikidata film titles and creators.

query_filter = """FILTER NOT EXISTS { ?work wdt:P577 [] }."""
dataframe = annual_query(query_filter)

for year in range(1890, 2030):
    query_filter = """
        ?work wdt:P577 ?publication.
        FILTER(YEAR(?publication) >= """+str(year)+""").
        FILTER(YEAR(?publication) < """+str(year+1)+""").
        """
    dataframe = pandas.concat([dataframe, annual_query(query_filter)])
        
save_path = pathlib.Path.home() / 'data' / 'wikidata-data' / 'wikidata-data.csv'
save_path.parents[0].mkdir(exist_ok=True)
dataframe.to_csv(save_path, index=False)

print(len(dataframe))
dataframe.sample(20)   

1690556


,creator_id,creator_name,work_id,work_name
9047,Q19629489,PATRICK PAROUX,Q1756026,HELL
18140,Q1045877,ORSO MARIA GUERRINI,Q3997676,Q3997676
4380,Q2959823,CHARLES MILLOT,Q1756571,UN MONDE NOUVEAU
959,Q2057449,PATRICK BREEN,Q82352,RADIO
13779,Q90035,ROLF THIELE,Q23960384,Q23960384
185,Q6284220,JOSEPH J. DOWLING,Q2449789,THE BARGAIN
3445,Q1117124,FRANCO GASPARRI,Q1242772,THE LAST MAN ON EARTH
6238,Q276278,LAETA KALOGRIDIS,Q42914959,THE HOUSE WITH A CLOCK IN ITS WALLS
4101,Q8026110,WINSTON MILLER,Q51879625,THE POWER OF A LIE
3913,Q106330,KLAUS POCHE,Q1540387,BORN IN '45
